In [ ]:
import pandas as pd

# ==============================
# 1. Cargar CSV desde GitHub
# ==============================

url = "https://raw.githubusercontent.com/iJahir/etl-data-pipeline-25168282020-/refs/heads/main/data/raw/D_departamentos.csv"

df = pd.read_csv(url)

print("Primeros registros:")
print(df.head())


# ==============================
# 2. Limpieza de datos
# ==============================

departamentos = df.copy()

# limpiar columnas
departamentos.columns = departamentos.columns.str.strip().str.lower()

# limpiar espacios
for col in departamentos.select_dtypes(include='object').columns:
    departamentos[col] = departamentos[col].astype(str).str.strip()

# eliminar duplicados
departamentos = departamentos.drop_duplicates()


# ==============================
# 3. Separar válidos y rechazados
# ==============================

validos = departamentos[
    (~departamentos["dept"].astype(str).str.contains("error|text")) &
    (~departamentos["nombre"].astype(str).str.contains("error|text")) &
    (departamentos["dept"].notna()) &
    (departamentos["nombre"].notna())
].copy()

rechazados = departamentos.drop(validos.index).copy()


# ==============================
# 4. Motivo de rechazo
# ==============================

def motivo(row):
    motivos = []

    if pd.isna(row["dept"]):
        motivos.append("dept_vacio")

    if pd.isna(row["nombre"]):
        motivos.append("nombre_vacio")

    if "error" in str(row["dept"]) or "error" in str(row["nombre"]):
        motivos.append("dato_error")

    if "text" in str(row["dept"]) or "text" in str(row["nombre"]):
        motivos.append("dato_texto")

    return ",".join(motivos)


rechazados["motivo_rechazo"] = rechazados.apply(motivo, axis=1)


# ==============================
# 5. Exportar
# ==============================

validos.to_csv("curated_departamentos.csv", index=False)
rechazados.to_csv("rejects_departamentos.csv", index=False)

print("Departamentos PROCESADO CORRECTAMENTE ✅")

Primeros registros:
    dept   nombre
0  error    error
1     64      NaN
2    NaN      NaN
3    NaN  text_31
4  error      NaN
Departamentos PROCESADO CORRECTAMENTE ✅


In [ ]:
from google.colab import files

files.download('curated_departamentos.csv')
files.download('rejects_departamentos.csv')
#Archivos descargado

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>